# 01 · Pythonic essentials: think in Python, not Java syntax
**⏱ 9 minutes · Comprehensions, missing values, strings and function arguments**

Python AI libraries frequently pass around objects that look like JSON: tool arguments, model outputs, retrieved documents and metadata. In Java you'd use Maps, Streams and DTOs. The same ideas are present in Python, but the idioms differ.

For this lesson, our fictional helpdesk has three tickets. **Nothing contacts an external service.**

## 1. From Java Streams to Python comprehensions

Imagine selecting the IDs of all high-priority tickets.

**Java:**

```java
List<Integer> highIds = tickets.stream()
    .filter(t -> "high".equals(t.get("priority")))
    .map(t -> (Integer) t.get("id"))
    .toList();
```

**Python:** read the expression as *make an ID for each ticket, if its priority is high*. Square brackets mean the result is a **list**.

**Predict:** which two IDs will be printed?

In [ ]:
tickets = [
    {"id": 101, "priority": "high", "topic": "account", "tags": ["login", "urgent"]},
    {"id": 102, "priority": "low", "topic": "billing", "tags": ["invoice"]},
    {"id": 103, "priority": "high", "topic": "technical", "tags": ["api", "urgent"]},
]
high_ids = [t["id"] for t in tickets if t["priority"] == "high"]
print(high_ids)
assert high_ids == [101, 103]

## 2. Maps and sets have comprehensions too

In Java, you know the difference between a `Map` and a `Set`. In Python:

- `{key: value for ...}` creates a **dict** (Map-like lookup).
- `{value for ...}` creates a **set**, removing duplicates.
- `[value for ...]` creates a **list**, preserving duplicates and order.

The second example has two loops. That's okay for a simple flatten operation, but for complex logic, **an ordinary loop is usually easier to review**.

In [ ]:
topics_by_id = {t["id"]: t["topic"] for t in tickets}
unique_tags = {tag for t in tickets for tag in t["tags"]}
print("Lookup:", topics_by_id)
print("Unique tags:", unique_tags)
assert topics_by_id[102] == "billing"
assert unique_tags == {"login", "urgent", "invoice", "api"}

### Your turn · 60 seconds

Build a dictionary mapping **only high-priority IDs to their topics**. Change the expression below rather than merely pressing Run. The cell contains a working reference so **Run All** stays safe.

<details><summary>Hint (open only if you need it)</summary>
Start with `{t["id"]: t["topic"] for t in tickets ...}` and filter by priority.
</details>

In [ ]:
# TODO: rewrite the expression to test yourself.
high_topics = {t["id"]: t["topic"] for t in tickets if t["priority"] == "high"}
assert high_topics == {101: "account", 103: "technical"}
print("Correct:", high_topics)

## 3. Java `null` versus Python `None`

A Java `map.get("owner")` returns `null` when the key isn't present. Python `dict.get("owner")` returns `None` in the same situation, whereas `dict["owner"]` raises `KeyError`.

**Important edge case:** `ticket.get("owner") or "unassigned"` treats an **empty string** as missing too. That isn't identical to `ticket.get("owner", "unassigned")`.

Predict the difference before running. This matters for partially populated model outputs.

In [ ]:
ticket = {"id": 104, "owner": ""}
print("get with default:", repr(ticket.get("owner", "unassigned")))
print("get with or:", repr(ticket.get("owner") or "unassigned"))
print("missing key:", ticket.get("unknown", "unassigned"))
assert ticket.get("owner") == ""

## 4. String manipulation and f-strings

**Java:**

```java
String message = String.format("Ticket %03d: %s", id, subject);
String[] words = subject.split(" ");
```

**Python:** `f"Ticket {id:03d}"` formats and zero-pads a number; `text.split()` splits on whitespace; `" ".join(words)` joins cleaned-up words. A slice `text[:6]` extracts up to but **not including** position 6.

These are useful for prompt templates, API logs and simple parsers.

In [ ]:
ticket_id = 7
subject = "Cannot   sign in"
normalized = " ".join(subject.split())
message = f"Ticket {ticket_id:03d}: {normalized}"
print(message)
print("First six characters:", normalized[:6])
assert message == "Ticket 007: Cannot sign in"

## 5. A Python-specific trap: mutable default arguments

In Java, creating a new list inside a method gives each call fresh state. Python evaluates a **default argument only once**, when the function is defined. Don't use a mutable list as a default.

Use `None` as a sentinel, then create a list *inside* the function. Here `*` makes `tags` keyword-only; `**options` unpacks a dictionary into keyword arguments.

In [ ]:
def add_tag(tag: str, *, tags: list[str] | None = None) -> list[str]:
    if tags is None:
        tags = []
    tags.append(tag)
    return tags

first = add_tag("first")
second = add_tag("second")
assert first == ["first"] and second == ["second"]
print(first, second)

options = {"tags": ["existing"]}
print(add_tag("new", **options))

### Checkpoint

Why are `ticket.get("owner", "unassigned")` and `ticket.get("owner") or "unassigned"` *not always equivalent*? If you understand that, you're reading Python rather than translating mechanically.

**Next:** [02 · OOP, typing and Pydantic](02_typing_pydantic.ipynb). More reference: [Java-to-Python fundamentals](../guides/02_java_python_map.md).